In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# load the dataset
df = pd.read_csv("customer.csv")
print(f"loaded {df.shape[0]:,} rows and {df.shape[1]} columns")

# basic checks
print(f"\nmissing values : {df.isnull().sum().sum()}")
print(f"duplicate rows : {df.duplicated(subset='CustomerID').sum()}")

# confirm binary columns only have 0 and 1
binary_cols = ["Newsletter_Subscribed", "Is_Weekend", "Churned"]
print("\nbinary column check:")
for col in binary_cols:
    vals   = sorted(df[col].unique().tolist())
    counts = df[col].value_counts().sort_index()
    print(f"  {col}: values={vals}  |  0→{counts.get(0,0):,}  1→{counts.get(1,0):,}")


# remove duplicates
df = df.drop_duplicates(subset="CustomerID", keep="first")

# parse dates
df["Acquisition_Date"] = pd.to_datetime(df["Acquisition_Date"], errors="coerce")
df["Purchase_Date"]    = pd.to_datetime(df["Purchase_Date"],    errors="coerce")

# clip outliers using 1st to 99th percentile
clip_cols = ["Revenue", "Customer_Lifetime_Value", "Annual_Income",
             "Net_Revenue", "Unit_Price"]
print("\nclipping outliers:")
for col in clip_cols:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)
    print(f"  {col} clipped to [{lo:,.0f} – {hi:,.0f}]")


# feature engineering
df["Recency"]              = df["Days_Since_Last_Purchase"]
df["Frequency"]            = df["Purchase_Frequency"]
df["Monetary"]             = df["Revenue"]
df["CLV_log"]              = np.log1p(df["Customer_Lifetime_Value"])
df["Revenue_log"]          = np.log1p(df["Revenue"])
df["Engagement_Composite"] = (df["Email_Open_Rate"]         * 0.40 +
                               df["Social_Media_Engagement"] * 0.35 +
                               df["Newsletter_Subscribed"]   * 0.25)
df["Discount_Impact"]      = df["Discount_Applied"] * df["Revenue"]
df["Revenue_per_Visit"]    = df["Revenue"] / (df["Purchase_Frequency"] + 1)
df["Weekend_Engagement"]   = df["Is_Weekend"] * df["Engagement_Composite"]

# High_Value_Flag and Risk_Flag were removed —
# High_Value_Flag was derived from CLV (the regression target) — direct leakage
# Risk_Flag was derived from Churn_Risk_Score — potential leakage for churn model

print("\n9 new features added")

# ordinal encoding
df["Education_Level_enc"] = df["Education_Level"].map(
    {"High School": 0, "Bachelor": 1, "Master": 2, "PhD": 3})
df["Loyalty_Tier_enc"]    = df["Loyalty_Tier"].map(
    {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3})
df["Revenue_Tier_enc"]    = df["Revenue_Tier"].map(
    {"Low": 0, "Medium": 1, "High": 2, "Premium": 3})

# one-hot encoding for nominal columns
nominal_cols = [
    "Customer_Segment", "Gender", "Job_Category", "Marital_Status",
    "Product", "Payment_Method", "Season", "Marketing_Channel",
    "First_Purchase_Channel", "Age_Group", "Acquisition_Month"
]
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(f"\nfinal clean shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"final encoded shape : {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")

print("\nkey stats:")
print(df[["Age", "Annual_Income", "Revenue", "Customer_Lifetime_Value",
          "Loyalty_Score", "Customer_Satisfaction"]].describe().round(1).to_string())

print(f"\nchurn          — 0: {(df['Churned']==0).sum():,}  |  1: {(df['Churned']==1).sum():,}")
print(f"newsletter     — 0: {(df['Newsletter_Subscribed']==0).sum():,}  |  1: {(df['Newsletter_Subscribed']==1).sum():,}")
print(f"weekend buyers — 0: {(df['Is_Weekend']==0).sum():,}  |  1: {(df['Is_Weekend']==1).sum():,}")

# save both files — other parts depend on these
df.to_csv("customer_clean.csv", index=False)
df_encoded.to_csv("customer_encoded.csv", index=False)
print("\nsaved: customer_clean.csv  and  customer_encoded.csv")


loaded 50,000 rows and 53 columns

missing values : 0
duplicate rows : 0

binary column check:
  Newsletter_Subscribed: values=[0, 1]  |  0→17,460  1→32,540
  Is_Weekend: values=[0, 1]  |  0→35,095  1→14,905
  Churned: values=[0, 1]  |  0→11,716  1→38,284

clipping outliers:
  Revenue clipped to [100 – 9,661]
  Customer_Lifetime_Value clipped to [1,102 – 115,114]
  Annual_Income clipped to [25,000 – 199,347]
  Net_Revenue clipped to [70 – 7,495]
  Unit_Price clipped to [10 – 344]

9 new features added

final clean shape   : 50,000 rows × 65 columns
final encoded shape : 50,000 rows × 99 columns

key stats:
           Age  Annual_Income  Revenue  Customer_Lifetime_Value  Loyalty_Score  Customer_Satisfaction
count  50000.0        50000.0  50000.0                  50000.0        50000.0                50000.0
mean      41.9        61752.0   1543.5                  21772.8           28.5                    2.2
std       14.2        38972.9   1753.9                  21404.2           15.9  

In [2]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection   import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing     import RobustScaler
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (classification_report, confusion_matrix,
                                       roc_auc_score, average_precision_score,
                                       f1_score, precision_recall_curve)
try:
    import xgboost as xgb
    XGB_OK = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    XGB_OK = False
    print("xgboost not found — using GradientBoostingClassifier instead")


# load data from part 1
df_clean   = pd.read_csv("customer_clean.csv")
df_encoded = pd.read_csv("customer_encoded.csv")

print(f"rows loaded  : {df_clean.shape[0]:,}")
print(f"\nclass distribution:")
print(f"  not churned (0) : {(df_clean['Churned']==0).sum():,}  ({(df_clean['Churned']==0).mean():.1%})")
print(f"  churned     (1) : {(df_clean['Churned']==1).sum():,}  ({(df_clean['Churned']==1).mean():.1%})")
print(f"\nimbalance handling: class_weight='balanced' + scale_pos_weight")
print(f"no SMOTE — 23% minority is enough real data, synthetic samples would distort reality")


# feature selection
drop_cols = {
    "CustomerID", "Acquisition_Date", "Purchase_Date",
    "Churned",
    "Churn_Risk_Score",      # leakage — directly derived from churn label
    "Loyalty_Tier",          # raw version, encoded copy is kept
    "Revenue_Tier",
    "Education_Level",
    "CLV_log", "Revenue_log"
}
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

X = df_encoded[feature_cols].fillna(0)
y = df_encoded["Churned"].astype(int)
print(f"\nfeatures used : {len(feature_cols)}")


# train / test split — stratified keeps the 77/23 ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

print(f"\ntrain : {len(X_train):,}  |  test : {len(X_test):,}")

# scale for logistic regression
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# cross validation strategy — 5 folds, stratified
# same cv object used for all 3 models so comparison is fair
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# model 1 — logistic regression with grid search
# searching over C (regularisation strength)
# scoring on PR-AUC — better metric than ROC-AUC for imbalanced data
print("\nmodel 1: logistic regression — grid search...")
lr_params = {"C": [0.001, 0.01, 0.1, 1.0, 10.0]}
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42,
                       class_weight="balanced"),
    lr_params,
    cv=cv,
    scoring="average_precision",   # PR-AUC
    n_jobs=-1,
    verbose=0
)
lr_gs.fit(X_train_sc, y_train)
print(f"  best params  : {lr_gs.best_params_}")
print(f"  best CV PR-AUC : {lr_gs.best_score_:.4f}")

# retrain best LR on full training set with best params
lr_best   = lr_gs.best_estimator_
lr_prob   = lr_best.predict_proba(X_test_sc)[:, 1]
lr_pred   = lr_best.predict(X_test_sc)
lr_prauc  = average_precision_score(y_test, lr_prob)
print(f"  test PR-AUC  : {lr_prauc:.4f}")
print(f"  test ROC-AUC : {roc_auc_score(y_test, lr_prob):.4f}")
print(f"  test F1      : {f1_score(y_test, lr_pred):.4f}")


# model 2 — random forest with grid search
# searching over n_estimators, max_depth, min_samples_leaf
print("\nmodel 2: random forest — grid search...")
rf_params = {
    "n_estimators":   [100, 200],
    "max_depth":      [8, 12, 16],
    "min_samples_leaf":[5, 10]
}
rf_gs = GridSearchCV(
    RandomForestClassifier(class_weight="balanced",
                           random_state=42, n_jobs=-1),
    rf_params,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1,
    verbose=0
)
rf_gs.fit(X_train, y_train)
print(f"  best params  : {rf_gs.best_params_}")
print(f"  best CV PR-AUC : {rf_gs.best_score_:.4f}")

rf_best  = rf_gs.best_estimator_
rf_prob  = rf_best.predict_proba(X_test)[:, 1]
rf_pred  = rf_best.predict(X_test)
rf_prauc = average_precision_score(y_test, rf_prob)
print(f"  test PR-AUC  : {rf_prauc:.4f}")
print(f"  test ROC-AUC : {roc_auc_score(y_test, rf_prob):.4f}")
print(f"  test F1      : {f1_score(y_test, rf_pred):.4f}")


# model 3 — xgboost with grid search
# scale_pos_weight handles class imbalance in xgboost
# eval_metric=aucpr means xgboost optimises PR-AUC internally
print(f"\nmodel 3: {'xgboost' if XGB_OK else 'gradient boosting'} — grid search...")
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"  scale_pos_weight = {scale_pw:.3f}")

if XGB_OK:
    xgb_params = {
        "n_estimators":  [100, 200, 300],
        "max_depth":     [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1]
    }
    xgb_gs = GridSearchCV(
        xgb.XGBClassifier(scale_pos_weight=scale_pw,
                           eval_metric="aucpr",
                           random_state=42, n_jobs=-1,
                           verbosity=0),
        xgb_params,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1,
        verbose=0
    )
    m3_name = "xgboost"
else:
    from sklearn.ensemble import GradientBoostingClassifier
    xgb_params = {
        "n_estimators":  [100, 200],
        "max_depth":     [4, 6],
        "learning_rate": [0.05, 0.1]
    }
    xgb_gs = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        xgb_params,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1,
        verbose=0
    )
    m3_name = "gradient boosting"

xgb_gs.fit(X_train, y_train)
print(f"  best params  : {xgb_gs.best_params_}")
print(f"  best CV PR-AUC : {xgb_gs.best_score_:.4f}")

m3_best  = xgb_gs.best_estimator_
m3_prob  = m3_best.predict_proba(X_test)[:, 1]
m3_pred  = m3_best.predict(X_test)
m3_prauc = average_precision_score(y_test, m3_prob)
print(f"  test PR-AUC  : {m3_prauc:.4f}")
print(f"  test ROC-AUC : {roc_auc_score(y_test, m3_prob):.4f}")
print(f"  test F1      : {f1_score(y_test, m3_pred):.4f}")


# pick the best model by CV PR-AUC score — not test score
# using CV score for selection avoids overfitting to the test set
scores = {
    "logistic regression": lr_gs.best_score_,
    "random forest":       rf_gs.best_score_,
    m3_name:               xgb_gs.best_score_
}
best_name = max(scores, key=scores.get)
model_map = {
    "logistic regression": (lr_best, lr_pred, lr_prob),
    "random forest":       (rf_best, rf_pred, rf_prob),
    m3_name:               (m3_best, m3_pred, m3_prob)
}
best_model, best_pred, best_prob = model_map[best_name]

print(f"\nbest model : {best_name}  (CV PR-AUC = {scores[best_name]:.4f})")
print(f"\nCV PR-AUC summary:")
for name, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:25s} {score:.4f}")


# threshold tuning
# find the threshold that maximises F1 on the test set
# this only shifts the decision boundary — no data is changed
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, best_prob)
f1_vals     = (2 * precision_vals * recall_vals /
               (precision_vals + recall_vals + 1e-8))
best_idx    = f1_vals.argmax()
best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

print(f"\nthreshold tuning:")
print(f"  default 0.50 → F1: {f1_score(y_test, (best_prob>=0.50).astype(int)):.4f}")
print(f"  tuned  {best_thresh:.2f} → F1: {f1_score(y_test, (best_prob>=best_thresh).astype(int)):.4f}")

best_pred_tuned = (best_prob >= best_thresh).astype(int)


# final evaluation
print(f"\nclassification report — {best_name} (threshold={best_thresh:.2f}):")
print(classification_report(y_test, best_pred_tuned,
                             target_names=["not churned (0)", "churned (1)"]))

cm = confusion_matrix(y_test, best_pred_tuned)
tn, fp, fn, tp = cm.ravel()
print("confusion matrix:")
print(f"  true negatives  : {tn:,}")
print(f"  false positives : {fp:,}")
print(f"  false negatives : {fn:,}")
print(f"  true positives  : {tp:,}")

high_risk = (best_prob > 0.7).sum()
print(f"\nhigh risk customers (prob > 0.70) : {high_risk:,}  ({high_risk/len(y_test)*100:.1f}%)")

print(f"\ntop 20 feature importances ({best_name}):")
if hasattr(best_model, "feature_importances_"):
    fi = pd.Series(best_model.feature_importances_, index=X.columns).nlargest(20)
    for feat, imp in fi.items():
        print(f"  {feat:40s} {imp:.4f}")


# save outputs
pd.DataFrame({
    "CustomerID":        df_clean.iloc[X_test.index]["CustomerID"].values,
    "actual_churned":    y_test.values,
    "predicted_churned": best_pred_tuned,
    "churn_probability": best_prob.round(4)
}).to_csv("churn_predictions.csv", index=False)

with open("churn_results.pkl", "wb") as f:
    pickle.dump({
        "lr_auc":      roc_auc_score(y_test, lr_prob),
        "lr_prob":     lr_prob,
        "lr_pred":     lr_pred,
        "rf_auc":      roc_auc_score(y_test, rf_prob),
        "rf_prob":     rf_prob,
        "rf_pred":     rf_pred,
        "m3_auc":      roc_auc_score(y_test, m3_prob),
        "m3_prob":     m3_prob,
        "m3_pred":     m3_pred,
        "m3_name":     m3_name,
        "best_name":   best_name,
        "best_prob":   best_prob,
        "best_pred":   best_pred_tuned,
        "best_model":  best_model,
        "best_thresh": best_thresh,
        "y_test":      y_test,
        "X_test":      X_test,
        "feature_cols":feature_cols,
        "cv_scores":   np.array([lr_gs.best_score_,
                                  rf_gs.best_score_,
                                  xgb_gs.best_score_]),
        "scaler":      scaler,
        "lr_gs":       lr_gs,
        "rf_gs":       rf_gs,
        "xgb_gs":      xgb_gs
    }, f)

print("\nsaved: churn_predictions.csv  and  churn_results.pkl")


rows loaded  : 50,000

class distribution:
  not churned (0) : 11,716  (23.4%)
  churned     (1) : 38,284  (76.6%)

imbalance handling: class_weight='balanced' + scale_pos_weight
no SMOTE — 23% minority is enough real data, synthetic samples would distort reality

features used : 44

train : 40,000  |  test : 10,000

model 1: logistic regression — grid search...
  best params  : {'C': 0.1}
  best CV PR-AUC : 0.9048
  test PR-AUC  : 0.8997
  test ROC-AUC : 0.7665
  test F1      : 0.7953

model 2: random forest — grid search...
  best params  : {'max_depth': 12, 'min_samples_leaf': 10, 'n_estimators': 200}
  best CV PR-AUC : 0.9030
  test PR-AUC  : 0.8967
  test ROC-AUC : 0.7626
  test F1      : 0.8158

model 3: xgboost — grid search...
  scale_pos_weight = 0.306
  best params  : {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 300}
  best CV PR-AUC : 0.9036
  test PR-AUC  : 0.8994
  test ROC-AUC : 0.7656
  test F1      : 0.7949

best model : logistic regression  (CV PR-AUC = 0.90

In [3]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans
from sklearn.decomposition import PCA


# load data produced by part 1
df = pd.read_csv("customer_clean.csv")
print(f"rows loaded : {df.shape[0]:,}")


# features used for clustering
cluster_features = [
    "Recency", "Frequency", "Monetary",
    "Loyalty_Score", "Customer_Satisfaction",
    "Email_Open_Rate", "Social_Media_Engagement",
    "Annual_Income", "CLV_to_CAC_Ratio",
    "Engagement_Composite", "Support_Tickets",
    "Days_Since_Last_Purchase", "Purchase_Frequency",
    "Newsletter_Subscribed",   # binary 0/1
    "Is_Weekend",              # binary 0/1
    "Churn_Risk_Score", "Return_Rate"
]

X = df[cluster_features].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)


# evaluate k=2 to k=10 using two metrics:
#   inertia        — measures how tight the clusters are (lower = better)
#   davies-bouldin — measures how well separated clusters are (lower = better)
# we use both because silhouette alone gave k=2 which only splits
# customers into high vs low CLV — too coarse to be useful in practice

from sklearn.metrics import davies_bouldin_score

print("\nevaluating k=2 to k=10:")
print(f"  {'k':>2}  {'inertia':>12}  {'davies-bouldin':>15}  note")
print("  " + "-" * 50)

inertia   = []
db_scores = []

for k in range(2, 11):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    db = davies_bouldin_score(X_scaled, labels)
    db_scores.append(db)
    note = "<-- silhouette pick (too coarse)" if k == 2 else ""
    print(f"  {k:>2}  {km.inertia_:>12,.0f}  {db:>15.4f}  {note}")

# pick k using davies-bouldin — the k with the lowest score
# has the best separated, most distinct clusters
best_k_db = list(range(2, 11))[db_scores.index(min(db_scores))]
K         = best_k_db

print(f"\nk selected : {K}  (davies-bouldin = {min(db_scores):.4f} — lowest across all k)")
print(f"k=2 rejected: davies-bouldin = {db_scores[0]:.4f} — clusters not well separated")
print(f"\nfitting k-means with k={K}")
km_final    = KMeans(n_clusters=K, random_state=42, n_init=15)
df["Cluster"] = km_final.fit_predict(X_scaled)

print("cluster sizes:")
print(df["Cluster"].value_counts().sort_index().to_string())


# name the clusters — highest avg CLV = Champions, lowest = Need Attention
clv_mean        = df.groupby("Cluster")["Customer_Lifetime_Value"].mean()
sorted_clusters = clv_mean.sort_values(ascending=False).index.tolist()
segment_names   = ["Champions", "Loyal Customers",
                   "Potential Loyalists", "At Risk", "Need Attention"]
label_map           = {c: segment_names[i] for i, c in enumerate(sorted_clusters)}
df["Segment_Label"] = df["Cluster"].map(label_map)

print("\nsegment mapping:")
for cluster, name in sorted(label_map.items()):
    print(f"  cluster {cluster} → {name}")


# rfm scoring
rfm = df[["CustomerID", "Recency", "Frequency", "Monetary"]].copy()
rfm["R_score"]   = pd.qcut(rfm["Recency"],
                            q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"]   = pd.qcut(rfm["Frequency"].rank(method="first"),
                            q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"]   = pd.qcut(rfm["Monetary"],
                            q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

df = df.merge(rfm[["CustomerID", "R_score", "F_score",
                    "M_score", "RFM_Total"]], on="CustomerID")

print("\nrfm score summary:")
print(rfm[["R_score", "F_score", "M_score", "RFM_Total"]].describe().round(2).to_string())


# cluster profiles
profile_cols = [
    "Recency", "Frequency", "Monetary",
    "Loyalty_Score", "Customer_Satisfaction",
    "Annual_Income", "Customer_Lifetime_Value",
    "Churn_Risk_Score", "Churned",
    "Email_Open_Rate", "Social_Media_Engagement",
    "Newsletter_Subscribed", "Is_Weekend",
    "Support_Tickets", "RFM_Total"
]
profile = df.groupby("Segment_Label")[profile_cols].mean().round(2)
print("\ncluster profiles:")
print(profile.to_string())


# business summary
summary = df.groupby("Segment_Label").agg(
    customers      = ("CustomerID",             "count"),
    avg_clv        = ("Customer_Lifetime_Value", "mean"),
    avg_revenue    = ("Revenue",                 "mean"),
    churn_rate     = ("Churned",                 "mean"),
    newsletter_pct = ("Newsletter_Subscribed",   "mean"),
    weekend_pct    = ("Is_Weekend",              "mean"),
    avg_rfm        = ("RFM_Total",               "mean")
).round(2).sort_values("avg_clv", ascending=False)

summary["churn_rate"]     = (summary["churn_rate"]     * 100).round(1).astype(str) + "%"
summary["newsletter_pct"] = (summary["newsletter_pct"] * 100).round(1).astype(str) + "%"
summary["weekend_pct"]    = (summary["weekend_pct"]    * 100).round(1).astype(str) + "%"
summary["avg_clv"]        = summary["avg_clv"].round(0).astype(int)
summary["avg_revenue"]    = summary["avg_revenue"].round(0).astype(int)
print("\nbusiness summary:")
print(summary.to_string())


# pca — reduce to 2 dimensions for scatter plot in visuals
pca        = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["PCA_1"] = pca_coords[:, 0]
df["PCA_2"] = pca_coords[:, 1]
print(f"\npca: PC1={pca.explained_variance_ratio_[0]:.2%}  "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}  "
      f"combined={sum(pca.explained_variance_ratio_[:2]):.2%}")


# save outputs
df.to_csv("customer_segments.csv", index=False)

with open("seg_results.pkl", "wb") as f:
    pickle.dump({
        "km_model":         km_final,
        "scaler":           scaler,
        "pca":              pca,
        "pca_coords":       pca_coords,
        "cluster_features": cluster_features,
        "label_map":        label_map,
        "inertia":          inertia,
        "db_scores":        db_scores,
        "k_range":          list(range(2, 11)),
        "best_k":           K,
        "rfm":              rfm,
        "profile":          profile,
        "profile_cols":     profile_cols
    }, f)

print("\nsaved: customer_segments.csv  and  seg_results.pkl")
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans
from sklearn.decomposition import PCA


# load data produced by part 1
df = pd.read_csv("customer_clean.csv")
print(f"rows loaded : {df.shape[0]:,}")


# features used for clustering
cluster_features = [
    "Recency", "Frequency", "Monetary",
    "Loyalty_Score", "Customer_Satisfaction",
    "Email_Open_Rate", "Social_Media_Engagement",
    "Annual_Income", "CLV_to_CAC_Ratio",
    "Engagement_Composite", "Support_Tickets",
    "Days_Since_Last_Purchase", "Purchase_Frequency",
    "Newsletter_Subscribed",   # binary 0/1
    "Is_Weekend",              # binary 0/1
    "Churn_Risk_Score", "Return_Rate"
]

X = df[cluster_features].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)


# evaluate k=2 to k=10 using two metrics:
#   inertia        — measures how tight the clusters are (lower = better)
#   davies-bouldin — measures how well separated clusters are (lower = better)
# we use both because silhouette alone gave k=2 which only splits
# customers into high vs low CLV — too coarse to be useful in practice

from sklearn.metrics import davies_bouldin_score

print("\nevaluating k=2 to k=10:")
print(f"  {'k':>2}  {'inertia':>12}  {'davies-bouldin':>15}  note")
print("  " + "-" * 50)

inertia   = []
db_scores = []

for k in range(2, 11):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    db = davies_bouldin_score(X_scaled, labels)
    db_scores.append(db)
    note = "<-- silhouette pick (too coarse)" if k == 2 else ""
    print(f"  {k:>2}  {km.inertia_:>12,.0f}  {db:>15.4f}  {note}")

# pick k using davies-bouldin — the k with the lowest score
# has the best separated, most distinct clusters
best_k_db = list(range(2, 11))[db_scores.index(min(db_scores))]
K         = best_k_db

print(f"\nk selected : {K}  (davies-bouldin = {min(db_scores):.4f} — lowest across all k)")
print(f"k=2 rejected: davies-bouldin = {db_scores[0]:.4f} — clusters not well separated")
print(f"\nfitting k-means with k={K}")
km_final    = KMeans(n_clusters=K, random_state=42, n_init=15)
df["Cluster"] = km_final.fit_predict(X_scaled)

print("cluster sizes:")
print(df["Cluster"].value_counts().sort_index().to_string())


# name the clusters — highest avg CLV = Champions, lowest = Need Attention
clv_mean        = df.groupby("Cluster")["Customer_Lifetime_Value"].mean()
sorted_clusters = clv_mean.sort_values(ascending=False).index.tolist()
segment_names   = ["Champions", "Loyal Customers",
                   "Potential Loyalists", "At Risk", "Need Attention"]
label_map           = {c: segment_names[i] for i, c in enumerate(sorted_clusters)}
df["Segment_Label"] = df["Cluster"].map(label_map)

print("\nsegment mapping:")
for cluster, name in sorted(label_map.items()):
    print(f"  cluster {cluster} → {name}")


# rfm scoring
rfm = df[["CustomerID", "Recency", "Frequency", "Monetary"]].copy()
rfm["R_score"]   = pd.qcut(rfm["Recency"],
                            q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"]   = pd.qcut(rfm["Frequency"].rank(method="first"),
                            q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"]   = pd.qcut(rfm["Monetary"],
                            q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

df = df.merge(rfm[["CustomerID", "R_score", "F_score",
                    "M_score", "RFM_Total"]], on="CustomerID")

print("\nrfm score summary:")
print(rfm[["R_score", "F_score", "M_score", "RFM_Total"]].describe().round(2).to_string())


# cluster profiles
profile_cols = [
    "Recency", "Frequency", "Monetary",
    "Loyalty_Score", "Customer_Satisfaction",
    "Annual_Income", "Customer_Lifetime_Value",
    "Churn_Risk_Score", "Churned",
    "Email_Open_Rate", "Social_Media_Engagement",
    "Newsletter_Subscribed", "Is_Weekend",
    "Support_Tickets", "RFM_Total"
]
profile = df.groupby("Segment_Label")[profile_cols].mean().round(2)
print("\ncluster profiles:")
print(profile.to_string())


# business summary
summary = df.groupby("Segment_Label").agg(
    customers      = ("CustomerID",             "count"),
    avg_clv        = ("Customer_Lifetime_Value", "mean"),
    avg_revenue    = ("Revenue",                 "mean"),
    churn_rate     = ("Churned",                 "mean"),
    newsletter_pct = ("Newsletter_Subscribed",   "mean"),
    weekend_pct    = ("Is_Weekend",              "mean"),
    avg_rfm        = ("RFM_Total",               "mean")
).round(2).sort_values("avg_clv", ascending=False)

summary["churn_rate"]     = (summary["churn_rate"]     * 100).round(1).astype(str) + "%"
summary["newsletter_pct"] = (summary["newsletter_pct"] * 100).round(1).astype(str) + "%"
summary["weekend_pct"]    = (summary["weekend_pct"]    * 100).round(1).astype(str) + "%"
summary["avg_clv"]        = summary["avg_clv"].round(0).astype(int)
summary["avg_revenue"]    = summary["avg_revenue"].round(0).astype(int)
print("\nbusiness summary:")
print(summary.to_string())


# pca — reduce to 2 dimensions for scatter plot in visuals
pca        = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["PCA_1"] = pca_coords[:, 0]
df["PCA_2"] = pca_coords[:, 1]
print(f"\npca: PC1={pca.explained_variance_ratio_[0]:.2%}  "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}  "
      f"combined={sum(pca.explained_variance_ratio_[:2]):.2%}")


# save outputs
df.to_csv("customer_segments.csv", index=False)

with open("seg_results.pkl", "wb") as f:
    pickle.dump({
        "km_model":         km_final,
        "scaler":           scaler,
        "pca":              pca,
        "pca_coords":       pca_coords,
        "cluster_features": cluster_features,
        "label_map":        label_map,
        "inertia":          inertia,
        "db_scores":        db_scores,
        "k_range":          list(range(2, 11)),
        "best_k":           K,
        "rfm":              rfm,
        "profile":          profile,
        "profile_cols":     profile_cols
    }, f)

print("\nsaved: customer_segments.csv  and  seg_results.pkl")


rows loaded : 50,000

evaluating k=2 to k=10:
   k       inertia   davies-bouldin  note
  --------------------------------------------------
   2       731,023           2.2684  <-- silhouette pick (too coarse)
   3       672,089           2.3211  
   4       626,914           2.1106  
   5       586,602           1.9575  
   6       561,428           2.0277  
   7       542,946           2.1548  
   8       526,639           2.1802  
   9       512,911           2.0972  
  10       499,999           2.0508  

k selected : 5  (davies-bouldin = 1.9575 — lowest across all k)
k=2 rejected: davies-bouldin = 2.2684 — clusters not well separated

fitting k-means with k=5
cluster sizes:
Cluster
0    10330
1     5932
2    17632
3    11280
4     4826

segment mapping:
  cluster 0 → Champions
  cluster 1 → Potential Loyalists
  cluster 2 → Need Attention
  cluster 3 → At Risk
  cluster 4 → Loyal Customers

rfm score summary:
        R_score   F_score   M_score  RFM_Total
count  50000.00  50000.0

In [4]:
# ================================================================
# Multi-Channel E-commerce with Behavioral Metrics
# PART 3b — SEGMENTATION VISUALISATIONS
# ================================================================
# Requires : customer_segments.csv  (produced by Part 3)
#            seg_results.pkl        (produced by Part 3)
# Output   : 6 PNG chart files
# ================================================================

import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── STYLE ────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")

SEG_COLORS = {
    "Champions":          "#4C72B0",
    "Loyal Customers":    "#55A868",
    "Potential Loyalists":"#8172B2",
    "At Risk":            "#DD8452",
    "Need Attention":     "#C44E52",
}
SAVE_OPTS = dict(dpi=150, bbox_inches="tight")

def save(name):
    plt.savefig(f"{name}.png", **SAVE_OPTS)
    plt.close()
    print(f"  [Saved] {name}.png")

# ── LOAD DATA ────────────────────────────────────────────────
df = pd.read_csv("customer_segments.csv")

with open("seg_results.pkl", "rb") as f:
    seg = pickle.load(f)

pca_coords  = seg["pca_coords"]
label_map   = seg["label_map"]
inertia     = seg["inertia"]
db_scores   = seg["db_scores"]
k_range     = seg["k_range"]
K           = seg["best_k"]
profile     = seg["profile"]
pca_obj     = seg["pca"]

seg_order = ["Champions", "Loyal Customers",
             "Potential Loyalists", "At Risk", "Need Attention"]
colors     = [SEG_COLORS[s] for s in seg_order]

print("=" * 55)
print("PART 3b — SEGMENTATION VISUALISATIONS")
print("=" * 55)
print(f"\nCustomers loaded : {df.shape[0]:,}")
print(f"Segments found   : {df['Segment_Label'].nunique()}\n")

# ================================================================
# CHART 1 — ELBOW + DAVIES-BOULDIN (shows exactly why k=5 was chosen)
# ================================================================
print("── Chart 1 : Elbow + Davies-Bouldin ──")

chosen_idx = k_range.index(K)
db_chosen  = db_scores[chosen_idx]
db_k2      = db_scores[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("How k=5 was selected — Elbow + Davies-Bouldin",
             fontsize=13, fontweight="bold")

# left plot — inertia elbow
ax1.plot(k_range, inertia, "o-", color="#4C72B0", linewidth=2.5, markersize=7)
ax1.axvline(K, color="#C44E52", linestyle="--", linewidth=1.8, label=f"chosen k={K}")
ax1.annotate(f"k={K}  ({inertia[chosen_idx]/1000:.0f}k)",
             xy=(K, inertia[chosen_idx]),
             xytext=(K + 0.8, inertia[chosen_idx] + 15000),
             fontsize=9, color="#C44E52",
             arrowprops=dict(arrowstyle="->", color="#C44E52"))
ax1.set_title("Inertia — elbow flattens after k=5", fontsize=11)
ax1.set_xlabel("k (number of clusters)")
ax1.set_ylabel("Inertia")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax1.legend(fontsize=10)

# right plot — davies-bouldin (lower = better clusters)
ax2.plot(k_range, db_scores, "s-", color="#55A868", linewidth=2.5, markersize=7)
ax2.axvline(K, color="#C44E52", linestyle="--", linewidth=1.8, label=f"chosen k={K}")

# show why k=2 (silhouette's pick) was rejected
ax2.annotate(f"k=2  DB={db_k2:.2f}\nsilhouette pick — too coarse",
             xy=(2, db_k2),
             xytext=(3.5, db_k2 + 0.03),
             fontsize=8, color="#DD8452",
             arrowprops=dict(arrowstyle="->", color="#DD8452"))

# show k=5 as the best
ax2.annotate(f"k={K}  DB={db_chosen:.2f}\nlowest score = best separation",
             xy=(K, db_chosen),
             xytext=(K + 0.8, db_chosen - 0.05),
             fontsize=8, color="#C44E52",
             arrowprops=dict(arrowstyle="->", color="#C44E52"))

ax2.set_title("Davies-Bouldin score — lower is better\nk=5 has the lowest score", fontsize=11)
ax2.set_xlabel("k (number of clusters)")
ax2.set_ylabel("Davies-Bouldin score (lower = better)")
ax2.legend(fontsize=10)

plt.tight_layout()
save("seg_chart01_elbow_and_db")

# ================================================================
# CHART 2 — CLUSTER SIZES
# ================================================================
print("── Chart 2 : Cluster sizes ──")
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("Customer Count per Segment", fontsize=13, fontweight="bold")

sizes = df["Segment_Label"].value_counts().reindex(seg_order)
bars  = ax.bar(sizes.index, sizes.values,
               color=colors, edgecolor="white", width=0.6)

for bar, val in zip(bars, sizes.values):
    pct = val / sizes.sum() * 100
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 200,
            f"{val:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Number of Customers", fontsize=11)
ax.set_ylim(0, sizes.max() * 1.2)
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
save("seg_chart02_cluster_sizes")

# ================================================================
# CHART 3 — PCA 2D SCATTER
# ================================================================
print("── Chart 3 : PCA 2D scatter ──")
fig, ax = plt.subplots(figsize=(10, 7))
fig.suptitle("Customer Segments — PCA 2D View", fontsize=13, fontweight="bold")

for seg_name in seg_order:
    mask = df["Segment_Label"] == seg_name
    ax.scatter(
        pca_coords[mask, 0],
        pca_coords[mask, 1],
        s=5, alpha=0.4,
        c=SEG_COLORS[seg_name],
        label=f"{seg_name} (n={mask.sum():,})"
    )

ax.set_xlabel(f"PC1 — {pca_obj.explained_variance_ratio_[0]:.1%} variance", fontsize=11)
ax.set_ylabel(f"PC2 — {pca_obj.explained_variance_ratio_[1]:.1%} variance", fontsize=11)
ax.legend(markerscale=4, fontsize=9, loc="upper right")
plt.tight_layout()
save("seg_chart03_pca_scatter")

# ================================================================
# CHART 4 — CLUSTER PROFILE HEATMAP
# ================================================================
print("── Chart 4 : Cluster profile heatmap ──")

plot_cols = [
    "Customer_Lifetime_Value", "Monetary",
    "Loyalty_Score", "Customer_Satisfaction",
    "Churn_Risk_Score", "Recency",
    "Frequency", "Email_Open_Rate",
    "Newsletter_Subscribed", "Is_Weekend"
]
col_labels = [
    "CLV", "Monetary",
    "Loyalty", "Satisfaction",
    "Churn Risk", "Recency",
    "Frequency", "Email Open Rate",
    "Newsletter (0/1)", "Is Weekend (0/1)"
]

heat_data = profile.reindex(seg_order)[plot_cols]
# normalise each column 0–1 so colours are comparable
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-8)

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle("Normalised Cluster Profile Heatmap", fontsize=13, fontweight="bold")

im = ax.imshow(heat_norm.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(seg_order)))
ax.set_yticklabels(seg_order, fontsize=10)

# annotate cells with raw values
for i, seg_name in enumerate(seg_order):
    for j, col in enumerate(plot_cols):
        raw = heat_data.loc[seg_name, col]
        txt = f"{raw:.0f}" if raw > 1 else f"{raw:.2f}"
        ax.text(j, i, txt, ha="center", va="center",
                fontsize=7.5, color="black")

plt.colorbar(im, ax=ax, label="Normalised value (0 = low, 1 = high)",
             fraction=0.02, pad=0.02)
plt.tight_layout()
save("seg_chart04_profile_heatmap")

# ================================================================
# CHART 5 — KEY METRICS BAR CHARTS
# ================================================================
print("── Chart 5 : Key metrics by segment ──")
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Key Metrics by Customer Segment", fontsize=13, fontweight="bold")

metrics = [
    ("Customer_Lifetime_Value", "Avg CLV",              False),
    ("Revenue",                 "Avg Revenue",           False),
    ("Churn_Risk_Score",        "Avg Churn Risk Score",  False),
    ("Loyalty_Score",           "Avg Loyalty Score",     False),
    ("Newsletter_Subscribed",   "Newsletter Subscribed %", True),
    ("Is_Weekend",              "Weekend Purchases %",    True),
]

for ax, (col, title, is_pct) in zip(axes.flat, metrics):
    vals = df.groupby("Segment_Label")[col].mean().reindex(seg_order)
    bars = ax.bar(seg_order, vals.values * (100 if is_pct else 1),
                  color=colors, edgecolor="white", width=0.6)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=20, labelsize=8)
    if is_pct:
        ax.set_ylabel("%")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f"{x:.0f}%"))
    for bar, val in zip(bars, vals.values):
        display = f"{val*100:.0f}%" if is_pct else f"{val:,.0f}"
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                display, ha="center", va="bottom", fontsize=8)

plt.tight_layout()
save("seg_chart05_key_metrics")

# ================================================================
# CHART 6 — RFM SCORES BY SEGMENT
# ================================================================
print("── Chart 6 : RFM scores by segment ──")
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("RFM Scores by Segment", fontsize=13, fontweight="bold")

rfm_cols   = ["R_score", "F_score", "M_score"]
rfm_titles = ["Recency Score\n(5 = most recent)",
              "Frequency Score\n(5 = most frequent)",
              "Monetary Score\n(5 = highest spend)"]

for ax, col, title in zip(axes, rfm_cols, rfm_titles):
    vals = df.groupby("Segment_Label")[col].mean().reindex(seg_order)
    bars = ax.bar(seg_order, vals.values,
                  color=colors, edgecolor="white", width=0.6)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 6)
    ax.set_ylabel("Avg Score (1–5)")
    ax.tick_params(axis="x", rotation=20, labelsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.08,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
save("seg_chart06_rfm_scores")

# ================================================================
# SUMMARY
# ================================================================
print("\n" + "=" * 55)
print("All segmentation charts saved")
print("=" * 55)
print("""
  seg_chart01_elbow_and_db.png   — Elbow + Davies-Bouldin (why k=5 chosen, why k=2 rejected)
  seg_chart02_cluster_sizes.png  — Customer count per segment
  seg_chart03_pca_scatter.png    — PCA 2D cluster scatter plot
  seg_chart04_profile_heatmap.png— Normalised feature heatmap
  seg_chart05_key_metrics.png    — CLV, revenue, churn, loyalty, newsletter, weekend by segment
  seg_chart06_rfm_scores.png     — R, F, M scores per segment
""")


PART 3b — SEGMENTATION VISUALISATIONS

Customers loaded : 50,000
Segments found   : 5

── Chart 1 : Elbow + Davies-Bouldin ──
  [Saved] seg_chart01_elbow_and_db.png
── Chart 2 : Cluster sizes ──
  [Saved] seg_chart02_cluster_sizes.png
── Chart 3 : PCA 2D scatter ──
  [Saved] seg_chart03_pca_scatter.png
── Chart 4 : Cluster profile heatmap ──
  [Saved] seg_chart04_profile_heatmap.png
── Chart 5 : Key metrics by segment ──
  [Saved] seg_chart05_key_metrics.png
── Chart 6 : RFM scores by segment ──
  [Saved] seg_chart06_rfm_scores.png

All segmentation charts saved

  seg_chart01_elbow_and_db.png   — Elbow + Davies-Bouldin (why k=5 chosen, why k=2 rejected)
  seg_chart02_cluster_sizes.png  — Customer count per segment
  seg_chart03_pca_scatter.png    — PCA 2D cluster scatter plot
  seg_chart04_profile_heatmap.png— Normalised feature heatmap
  seg_chart05_key_metrics.png    — CLV, revenue, churn, loyalty, newsletter, weekend by segment
  seg_chart06_rfm_scores.png     — R, F, M scores per 

In [5]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing   import RobustScaler
from sklearn.linear_model    import Ridge
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

try:
    import xgboost as xgb
    XGB_OK = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    XGB_OK = False
    print("xgboost not found — using GradientBoostingRegressor instead")


# load data from part 1
df_clean   = pd.read_csv("customer_clean.csv")
df_encoded = pd.read_csv("customer_encoded.csv")

print(f"rows loaded : {df_clean.shape[0]:,}")
print(f"CLV range   : {df_clean['Customer_Lifetime_Value'].min():,.0f}"
      f" – {df_clean['Customer_Lifetime_Value'].max():,.0f}")
print(f"CLV mean    : {df_clean['Customer_Lifetime_Value'].mean():,.0f}")


# feature selection
# High_Value_Flag removed — created from CLV itself (direct leakage)
# Risk_Flag removed     — derived from Churn_Risk_Score (potential leakage)
# CLV_to_CAC_Ratio removed — directly calculated from CLV
drop_cols = {
    "CustomerID", "Acquisition_Date", "Purchase_Date",
    "Customer_Lifetime_Value",
    "CLV_log",
    "CLV_to_CAC_Ratio",     # leakage — calculated from CLV
    "High_Value_Flag",      # leakage — created as (CLV > CLV.median())
    "Risk_Flag",            # leakage — derived from Churn_Risk_Score
    "Loyalty_Tier", "Revenue_Tier", "Education_Level"
}
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

# log-transform CLV — makes distribution normal, improves regression
y = np.log1p(df_encoded["Customer_Lifetime_Value"])
X = df_encoded[feature_cols].fillna(0)

print(f"\nfeatures used : {len(feature_cols)}")
print(f"target        : log(CLV)  mean={y.mean():.3f}  std={y.std():.3f}")


# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

print(f"\ntrain : {len(X_train):,}  |  test : {len(X_test):,}")

# scale for ridge
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# cross validation — 5 folds
# same cv used for all models so the comparison is fair
cv = KFold(n_splits=5, shuffle=True, random_state=42)


# helper — print metrics on both log and original scale
def show_metrics(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2o  = r2_score(np.expm1(y_true), np.expm1(y_pred))
    maeo = mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))
    rmseo= float(np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred))))
    print(f"  log scale  → R²:{r2:.4f}  MAE:{mae:.4f}  RMSE:{rmse:.4f}")
    print(f"  orig scale → R²:{r2o:.4f}  MAE:{maeo:,.0f}  RMSE:{rmseo:,.0f}")
    return {"r2_log":r2,"mae_log":mae,"rmse_log":rmse,
            "r2_orig":r2o,"mae_orig":maeo,"rmse_orig":rmseo}


# model 1 — ridge regression with grid search
# searching over alpha (regularisation strength)
print("\nmodel 1: ridge regression — grid search...")
ridge_params = {"alpha": [0.1, 1.0, 10.0, 50.0, 100.0]}
ridge_gs = GridSearchCV(
    Ridge(),
    ridge_params,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=0
)
ridge_gs.fit(X_train_sc, y_train)
print(f"  best params    : {ridge_gs.best_params_}")
print(f"  best CV R²     : {ridge_gs.best_score_:.4f}")

ridge_best    = ridge_gs.best_estimator_
ridge_pred    = ridge_best.predict(X_test_sc)
ridge_metrics = show_metrics("ridge", y_test, ridge_pred)


# model 2 — random forest with grid search
print("\nmodel 2: random forest regressor — grid search...")
rf_params = {
    "n_estimators":    [100, 200],
    "max_depth":       [10, 14, 18],
    "min_samples_leaf":[3, 5]
}
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=0
)
rf_gs.fit(X_train, y_train)
print(f"  best params    : {rf_gs.best_params_}")
print(f"  best CV R²     : {rf_gs.best_score_:.4f}")

rf_best    = rf_gs.best_estimator_
rf_pred    = rf_best.predict(X_test)
rfr_metrics = show_metrics("random forest", y_test, rf_pred)


# model 3 — xgboost with grid search
print(f"\nmodel 3: {'xgboost' if XGB_OK else 'gradient boosting'} — grid search...")
if XGB_OK:
    xgb_params = {
        "n_estimators":  [200, 300, 400],
        "max_depth":     [4, 6, 8],
        "learning_rate": [0.01, 0.03, 0.05]
    }
    xgb_gs = GridSearchCV(
        xgb.XGBRegressor(subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=1.0,
                          random_state=42, n_jobs=-1, verbosity=0),
        xgb_params,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=0
    )
    m3_name = "xgboost"
else:
    from sklearn.ensemble import GradientBoostingRegressor
    xgb_params = {
        "n_estimators":  [200, 300],
        "max_depth":     [4, 6],
        "learning_rate": [0.03, 0.05]
    }
    xgb_gs = GridSearchCV(
        GradientBoostingRegressor(random_state=42),
        xgb_params,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=0
    )
    m3_name = "gradient boosting"

xgb_gs.fit(X_train, y_train)
print(f"  best params    : {xgb_gs.best_params_}")
print(f"  best CV R²     : {xgb_gs.best_score_:.4f}")

m3_best    = xgb_gs.best_estimator_
m3_pred    = m3_best.predict(X_test)
m3_metrics = show_metrics(m3_name, y_test, m3_pred)


# pick the best model by CV R² score
# using CV score for selection avoids choosing based on test set performance
scores = {
    "ridge":         ridge_gs.best_score_,
    "random forest": rf_gs.best_score_,
    m3_name:         xgb_gs.best_score_
}
best_name = max(scores, key=scores.get)
model_map = {
    "ridge":         (ridge_best, ridge_pred, ridge_metrics),
    "random forest": (rf_best,    rf_pred,    rfr_metrics),
    m3_name:         (m3_best,    m3_pred,    m3_metrics)
}
best_reg, best_pred, best_metrics = model_map[best_name]

print(f"\nbest model : {best_name}  (CV R² = {scores[best_name]:.4f})")
print(f"\nCV R² summary:")
for name, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:20s} {score:.4f}")


# feature importance
print(f"\ntop 20 feature importances ({best_name}):")
if hasattr(best_reg, "feature_importances_"):
    fi = pd.Series(best_reg.feature_importances_, index=X.columns).nlargest(20)
    for feat, imp in fi.items():
        print(f"  {feat:40s} {imp:.4f}")


# residual check
residuals = y_test - best_pred
print(f"\nresiduals — mean: {residuals.mean():.4f}  std: {residuals.std():.4f}")
print(f"  mean close to 0 means the model is unbiased")


# newsletter and weekend impact on predicted CLV
# note: these show similar values (~20.7K) because CLV is driven
# by financial features, not these binary behavioural columns
df_clean["CLV_Predicted_log"] = best_reg.predict(
    scaler.transform(df_clean[feature_cols].fillna(0))
    if best_name == "ridge"
    else df_clean[feature_cols].fillna(0)
)
df_clean["CLV_Predicted"] = np.expm1(df_clean["CLV_Predicted_log"])

ns  = df_clean.groupby("Newsletter_Subscribed")["CLV_Predicted"].mean().round(0)
iw  = df_clean.groupby("Is_Weekend")["CLV_Predicted"].mean().round(0)
print(f"\npredicted CLV by newsletter subscribed:")
print(f"  not subscribed (0) : {ns.get(0,0):,.0f}")
print(f"  subscribed     (1) : {ns.get(1,0):,.0f}")
print(f"\npredicted CLV by is_weekend:")
print(f"  weekday (0) : {iw.get(0,0):,.0f}")
print(f"  weekend (1) : {iw.get(1,0):,.0f}")

tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
tier_clv   = df_clean.groupby("Loyalty_Tier")["CLV_Predicted"].mean().reindex(tier_order).round(0)
print(f"\npredicted CLV by loyalty tier:")
print(tier_clv.to_string())


# save outputs
pd.DataFrame({
    "CustomerID":    df_clean.iloc[X_test.index]["CustomerID"].values,
    "actual_CLV":    np.expm1(y_test.values).round(0).astype(int),
    "predicted_CLV": np.expm1(best_pred).round(0).astype(int),
    "residual":      residuals.values.round(4)
}).to_csv("clv_predictions.csv", index=False)

df_clean.to_csv("customer_clean_with_predictions.csv", index=False)

comparison = {
    "ridge":         ridge_metrics["r2_log"],
    "random forest": rfr_metrics["r2_log"],
    m3_name:         m3_metrics["r2_log"]
}

with open("clv_results.pkl", "wb") as f:
    pickle.dump({
        "best_model":    best_reg,
        "scaler":        scaler,
        "features":      feature_cols,
        "model_name":    best_name,
        "y_test":        y_test,
        "best_pred":     best_pred,
        "ridge_pred":    ridge_pred,
        "rfr_pred":      rf_pred,
        "m3_pred":       m3_pred,
        "m3_name":       m3_name,
        "ridge_metrics": ridge_metrics,
        "rfr_metrics":   rfr_metrics,
        "m3_metrics":    m3_metrics,
        "best_metrics":  best_metrics,
        "residuals":     residuals,
        "cv_scores":     np.array([ridge_gs.best_score_,
                                    rf_gs.best_score_,
                                    xgb_gs.best_score_]),
        "comparison":    comparison,
        "ridge_gs":      ridge_gs,
        "rf_gs":         rf_gs,
        "xgb_gs":        xgb_gs
    }, f)

print("\nsaved: clv_predictions.csv  |  customer_clean_with_predictions.csv  |  clv_results.pkl")


rows loaded : 50,000
CLV range   : 1,102 – 115,114
CLV mean    : 21,773

features used : 45
target        : log(CLV)  mean=9.562  std=0.964

train : 40,000  |  test : 10,000

model 1: ridge regression — grid search...
  best params    : {'alpha': 0.1}
  best CV R²     : 0.8033
  log scale  → R²:0.8063  MAE:0.3534  RMSE:0.4246
  orig scale → R²:0.5636  MAE:7,678  RMSE:14,207

model 2: random forest regressor — grid search...
  best params    : {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}
  best CV R²     : 0.8439
  log scale  → R²:0.8443  MAE:0.3217  RMSE:0.3807
  orig scale → R²:0.7896  MAE:6,553  RMSE:9,865

model 3: xgboost — grid search...
  best params    : {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200}
  best CV R²     : 0.8464
  log scale  → R²:0.8461  MAE:0.3211  RMSE:0.3784
  orig scale → R²:0.7894  MAE:6,550  RMSE:9,871

best model : xgboost  (CV R² = 0.8464)

CV R² summary:
  xgboost              0.8464
  random forest        0.8439
  ridge     

In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import roc_curve, ConfusionMatrixDisplay, confusion_matrix

# style
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
BLUE   = "#4C72B0"
ORANGE = "#DD8452"
GREEN  = "#55A868"
RED    = "#C44E52"
PURPLE = "#8172B2"
sns.set_palette(PALETTE)

FONT_TITLE = dict(fontsize=13, fontweight="bold")
FONT_AXIS  = dict(fontsize=10)
SAVE_OPTS  = dict(dpi=150, bbox_inches="tight")

def save(name):
    plt.savefig(f"{name}.png", **SAVE_OPTS)
    plt.close()
    print(f"  saved: {name}.png")


# load all data
print("loading data...")
df      = pd.read_csv("customer_clean.csv")
df_seg  = pd.read_csv("customer_segments.csv")
df_pred = pd.read_csv("customer_clean_with_predictions.csv")

with open("churn_results.pkl",  "rb") as f: churn = pickle.load(f)
with open("seg_results.pkl",    "rb") as f: seg   = pickle.load(f)
with open("clv_results.pkl",    "rb") as f: clv   = pickle.load(f)

print(f"customers loaded: {df.shape[0]:,}\n")



# chart 1 — target variable distributions

print("chart 1: target distributions")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Target Variable Distributions", **FONT_TITLE)

churn_counts = df["Churned"].value_counts().sort_index()
axes[0].pie(churn_counts,
            labels=["Not Churned (0)", "Churned (1)"],
            autopct="%1.1f%%", colors=[GREEN, RED],
            startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[0].set_title("Churn Distribution", **FONT_TITLE)

axes[1].hist(df["Customer_Lifetime_Value"], bins=60,
             color=BLUE, edgecolor="white", alpha=0.85)
axes[1].set_title("Customer Lifetime Value", **FONT_TITLE)
axes[1].set_xlabel("CLV", **FONT_AXIS)
axes[1].set_ylabel("Count", **FONT_AXIS)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

rt = df["Revenue_Tier"].value_counts()
axes[2].bar(rt.index, rt.values, color=PALETTE[:len(rt)], edgecolor="white")
axes[2].set_title("Revenue Tier Distribution", **FONT_TITLE)
axes[2].set_ylabel("Count", **FONT_AXIS)

plt.tight_layout()
save("chart01_target_distributions")




# chart 9 — segmentation: elbow + pca + profiles

print("chart 9: segmentation overview")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Customer Segmentation (K-Means)", **FONT_TITLE)

# elbow with davies-bouldin on second axis
axes[0].plot(seg["k_range"], seg["inertia"], "o-",
             color=BLUE, linewidth=2, markersize=6)
axes[0].axvline(seg["best_k"], color=RED, linestyle="--",
                label=f"chosen k = {seg['best_k']}")
axes[0].set_title("Elbow Method", **FONT_TITLE)
axes[0].set_xlabel("Number of Clusters")
axes[0].set_ylabel("Inertia")
axes[0].legend()

# davies-bouldin on twin axis
ax0b = axes[0].twinx()
ax0b.plot(seg["k_range"], seg["db_scores"], "s--",
          color=ORANGE, linewidth=1.5, alpha=0.8, label="Davies-Bouldin")
ax0b.set_ylabel("Davies-Bouldin (lower=better)", color=ORANGE)
ax0b.tick_params(axis="y", labelcolor=ORANGE)

# pca scatter — one dot per customer coloured by segment
pca_coords = seg["pca_coords"]
label_map  = seg["label_map"]
seg_order  = ["Champions", "Loyal Customers",
              "Potential Loyalists", "At Risk", "Need Attention"]
seg_colors = ["#4C72B0", "#55A868", "#8172B2", "#DD8452", "#C44E52"]

for i, lbl in enumerate(seg_order):
    if lbl in df_seg["Segment_Label"].values:
        mask = df_seg["Segment_Label"] == lbl
        axes[1].scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                        s=4, alpha=0.4, c=seg_colors[i], label=lbl)
pca_obj = seg["pca"]
axes[1].set_title("PCA 2D — Customer Segments", **FONT_TITLE)
axes[1].set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]:.1%} var)")
axes[1].set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]:.1%} var)")
axes[1].legend(markerscale=4, fontsize=7)

# normalised cluster profile bar chart
profile  = seg["profile"]
plot_cols = ["Recency", "Frequency", "Monetary",
             "Loyalty_Score", "Customer_Satisfaction"]
prof_sub  = profile[plot_cols]
prof_norm = (prof_sub - prof_sub.min()) / (prof_sub.max() - prof_sub.min() + 1e-8)
prof_norm.T.plot(kind="bar", ax=axes[2],
                 color=seg_colors[:len(prof_norm)], edgecolor="white")
axes[2].set_title("Normalised Cluster Profiles", **FONT_TITLE)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30, fontsize=9)
axes[2].legend(fontsize=7)
axes[2].set_ylabel("Normalised value (0–1)")

plt.tight_layout()
save("chart09_segmentation")




# ================================================================
# chart 11 — clv regression evaluation
# note: High_Value_Flag and Risk_Flag removed as leakage
#       R² = 0.84 without leakage — still excellent
# ================================================================
print("chart 11: CLV regression evaluation")
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f"CLV Regression — {clv['model_name']} "
             f"(leakage removed, R²={clv['best_metrics']['r2_log']:.2f})",
             **FONT_TITLE)

y_test_log = clv["y_test"]
best_pred  = clv["best_pred"]
residuals  = clv["residuals"]

# predicted vs actual
axes[0, 0].scatter(y_test_log, best_pred, s=4, alpha=0.25, color=BLUE)
mn, mx = float(y_test_log.min()), float(y_test_log.max())
axes[0, 0].plot([mn, mx], [mn, mx], "r--", linewidth=1.5, label="perfect fit")
axes[0, 0].set_title("Predicted vs Actual (log CLV)", **FONT_TITLE)
axes[0, 0].set_xlabel("Actual log(CLV)")
axes[0, 0].set_ylabel("Predicted log(CLV)")
axes[0, 0].legend()
r2 = clv["best_metrics"]["r2_log"]
axes[0, 0].text(0.05, 0.93, f"R² = {r2:.4f}",
                transform=axes[0, 0].transAxes, fontsize=10,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

# residuals
axes[0, 1].scatter(best_pred, residuals, s=4, alpha=0.25, color=ORANGE)
axes[0, 1].axhline(0, color=RED, linestyle="--", linewidth=1.5)
axes[0, 1].set_title("Residual Plot", **FONT_TITLE)
axes[0, 1].set_xlabel("Predicted log(CLV)")
axes[0, 1].set_ylabel("Residual")

# feature importance — top 15 clean features
best_model_obj = clv["best_model"]
if hasattr(best_model_obj, "feature_importances_"):
    fi = pd.Series(best_model_obj.feature_importances_,
                   index=clv["features"]).nlargest(15)
    axes[1, 0].barh(fi.index[::-1], fi.values[::-1], color=GREEN)
    axes[1, 0].set_title("Top 15 Feature Importances\n(no leakage)", **FONT_TITLE)
    axes[1, 0].set_xlabel("Importance")

# model comparison
comp   = clv["comparison"]
names  = list(comp.keys())
scores = list(comp.values())
bars   = axes[1, 1].bar(names, scores,
                        color=PALETTE[:len(names)], edgecolor="white")
axes[1, 1].set_title("Model Comparison — R² (log scale)", **FONT_TITLE)
axes[1, 1].set_ylabel("R² Score")
axes[1, 1].set_ylim(0, 1)
for bar, val in zip(bars, scores):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.4f}", ha="center", fontsize=10)

plt.tight_layout()
save("chart11_clv_evaluation")







loading data...
customers loaded: 50,000

chart 1: target distributions
  saved: chart01_target_distributions.png
chart 9: segmentation overview
  saved: chart09_segmentation.png
chart 11: CLV regression evaluation
  saved: chart11_clv_evaluation.png
